# Test: PDF Translation Pipeline

Tests `Translator` and `PDFTranslator` from `pdf_translate.py` using `co020es.pdf`.

In [1]:
# Install dependencies if needed
!pip install pytesseract pymupdf Pillow ollama -q

## 1. Start Ollama and pull model

In [1]:
import subprocess, time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(3)
print("Ollama server started")

Ollama server started


In [2]:
!ollama pull translategemma

pulling manifest ⠙ pulling manifest ⠙ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠴ pulling manifest ⠧ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest 
pulling bdbf939b402e:   0% ▕                  ▏ 4.2 MB/3.3 GB                  pulling manifest 
pulling bdbf939b402e:   0% ▕                  ▏ 7.3 MB/3.3 GB                  pulling manifest 
pulling bdbf939b402e:   0% ▕                  ▏  14 MB/3.3 GB                  pulling manifest 
pulling bdbf939b402e:   1% ▕                  ▏  21 MB/3.3 GB                  pulling manifest 
pulling bdbf939b402e:   1% ▕                  ▏  26 MB/3.3 GB                  pulling manifest 
pulling bdbf939b402e:   1% ▕                  ▏  34 MB/3.3 GB                  pulling manifest 
pulling bdbf939b402e:   1% ▕                  ▏  42 MB/3.3 GB                  pulling manifest 
pulling bdbf939b402e:   1% ▕                  ▏  45 MB/3.3 GB                  pulling manifest 
pulling bdbf939b

## 2. Import classes

In [3]:
from pdf_translate import Translator, PDFTranslator

## 3. Test Translator standalone

In [5]:
t = Translator(source_lang="Spanish")  # defaults to translategemma
print(t.translate("Hola mundo, esto es una prueba."))

Hello world, this is a test.


## 4. Test PDF extraction (no translation)

In [8]:
pdf = PDFTranslator(t)
result = pdf.extract("co020es_p1.pdf")

print(f"Total pages: {len(result['pages'])}")
for p in result["pages"]:
    print(f"  Page {p['page']}: type={p['type']}, chars={len(p['text'])}")

Total pages: 1
  Page 1: type=text, chars=2809


In [9]:
# Preview extracted text from first page
print(result["pages"][0]["text"][:500])

SENADO DE LA REPUBLICA DE COLOMBIA
Última Actualización : Abril 1, 2005. 
PREAMBULO 
El pueblo de Colombia, 
en ejercicio de su poder soberano, representado por sus delegatarios a la Asamblea Nacional 
Constituyente, invocando la protección de Dios, y con el fin de fortalecer la unidad de la Nación y 
asegurar a sus integrantes la vida, la convivencia, el trabajo, la justicia, la igualdad, el 
conocimiento, la libertad y la paz, dentro de un marco jurídico, democrático y participativo que 
garan


## 5. Test page-by-page translation

In [10]:
translated_pages = pdf.translate("co020es_p1.pdf")

Extracted 1 pages (1 text, 0 OCR)
  Translating page 1/1...


In [11]:
# Inspect results page by page
for p in translated_pages:
    print(f"--- Page {p['page']} ({p['type']}) ---")
    print(p["translation"][:300])
    print()

--- Page 1 (text) ---
SENATE OF THE REPUBLIC OF COLOMBIA
Last Update: April 1, 2005.
PREAMBLE
The people of Colombia,
in the exercise of their sovereign power, represented by their delegates to the National Constituent Assembly, invoking the protection of God, and with the aim of strengthening the unity of the Nation and



In [12]:
!ollama ps

NAME                     ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
translategemma:latest    c49d986b0764    4.3 GB    100% GPU     4096       4 minutes from now    


## 6. Test translate_file (save to disk)

In [13]:
meta = pdf.translate_file("co020es_p1.pdf", "co020es_p1_translated.txt")
meta

Extracted 1 pages (1 text, 0 OCR)
  Translating page 1/1...
Saved translation to co020es_p1_translated.txt


{'input': 'co020es_p1.pdf',
 'output': 'co020es_p1_translated.txt',
 'total_pages': 1,
 'text_pages': 1,
 'ocr_pages': 0,
 'chars_in': 2809,
 'chars_out': 2741,
 'pages': [{'page': 1,
   'type': 'text',
   'source': 'SENADO DE LA REPUBLICA DE COLOMBIA\nÚltima Actualización : Abril 1, 2005. \nPREAMBULO \nEl pueblo de Colombia, \nen ejercicio de su poder soberano, representado por sus delegatarios a la Asamblea Nacional \nConstituyente, invocando la protección de Dios, y con el fin de fortalecer la unidad de la Nación y \nasegurar a sus integrantes la vida, la convivencia, el trabajo, la justicia, la igualdad, el \nconocimiento, la libertad y la paz, dentro de un marco jurídico, democrático y participativo que \ngarantice un orden político, económico y social justo, y comprometido a impulsar la integración de \nla comunidad latinoamericana, decreta, sanciona y promulga la siguiente: \nCONSTITUCION POLITICA DE COLOMBIA \nTITULO I. \nDE LOS PRINCIPIOS FUNDAMENTALES \nARTICULO 1. Colombia e

## 7. Cleanup

In [15]:
ollama_process.terminate()
!ollama stop translategemma 2>/dev/null
print("Ollama stopped")

Ollama stopped
